In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q pandas openpyxl librosa tqdm requests

from __future__ import annotations

import argparse
import re
import sys
import zipfile
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

try:
    import librosa
except ImportError as e:
    print("Установите: pip install librosa soundfile numpy", file=sys.stderr)
    raise SystemExit(1) from e

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".opus"}

SOUND_DICTIONARY_SOURCE_NENA = "Sound dictionary of NENA varieties spoken in Russia"
NO_PARTS_PLACEHOLDER = "no parts"
NO_PARTS_LENGTH_ZERO = "00:00:00"


def normalize_excel_text(val: object) -> str:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()


def strip_leading_jupyter_kernel_args(rest: list[str]) -> list[str]:
    r = list(rest)
    while len(r) >= 2 and r[0] == "-f":
        r = r[2:]
    return r


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def download_yandex_public(url: str, output_path: str | Path) -> None:
    base_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key="
    final_url = base_url + url
    r = requests.get(final_url, timeout=120)
    r.raise_for_status()
    payload = r.json()
    if "href" not in payload:
        raise RuntimeError(f"Неожиданный ответ Yandex Disk API: {payload}")
    download_url = payload["href"]
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(download_url, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        with open(out, "wb") as f:
            for chunk in tqdm(resp.iter_content(1024 * 256), desc="Скачивание", unit="chunk"):
                if chunk:
                    f.write(chunk)


def canonical_suffix(filename: str) -> str | None:
    stem = Path(filename).stem.lower()
    if stem.startswith("audio"):
        return stem[len("audio") :]
    if stem.startswith("text"):
        return stem[len("text") :]
    return None


def collect_ids_by_suffix(
    paths: list[Path],
) -> tuple[dict[str, str], dict[str, str]]:
    """suffix -> basename для audio- и text-файлов."""
    audio_map: dict[str, str] = {}
    text_map: dict[str, str] = {}
    for p in paths:
        if not p.is_file():
            continue
        suf = canonical_suffix(p.name)
        if suf is None:
            continue
        ext = p.suffix.lower()
        if ext == ".txt":
            if suf not in text_map:
                text_map[suf] = p.name
        elif ext in AUDIO_EXTS:
            if suf not in audio_map:
                audio_map[suf] = p.name
    return audio_map, text_map


def check_pairs_report(
    text_dir: Path,
    audio_dir: Path,
    *,
    heading: str | None = None,
) -> bool:
    txts = list(text_dir.glob("*.txt"))
    audios = [p for p in audio_dir.iterdir() if p.suffix.lower() in AUDIO_EXTS]
    au_map, txt_map = collect_ids_by_suffix(audios + txts)

    au_keys = set(au_map)
    txt_keys = set(txt_map)
    missing_audio = sorted(txt_keys - au_keys)
    missing_text = sorted(au_keys - txt_keys)
    all_paired = not missing_audio and not missing_text

    lines: list[str] = []
    if heading:
        lines.append(heading)
        lines.append("")
    lines.extend(
        [
            "Проверка соответствия text ↔ audio (по суффиксу имени после префикса audio-/text-):",
            f"  .txt файлов: {len(txts)}",
            f"  аудиофайлов: {len(audios)}",
            f"  уникальных суффиксов у текстов (пары по имени): {len(txt_keys)}",
            f"  уникальных суффиксов у аудио: {len(au_keys)}",
        ]
    )
    if all_paired:
        lines.append("  У каждого суффикса есть и текст, и аудио (строгая пара).")
    else:
        if missing_audio:
            lines.append(f"  Нет аудио для {len(missing_audio)} текстов (по суффиксу):")
            for s in missing_audio[:15]:
                lines.append(f"      {txt_map[s]}  (суффикс: {s})")
            if len(missing_audio) > 15:
                lines.append(f"      … ещё {len(missing_audio) - 15}")
        if missing_text:
            lines.append(f"  Нет текста для {len(missing_text)} аудио:")
            for s in missing_text[:15]:
                lines.append(f"      {au_map[s]}  (суффикс: {s})")
            if len(missing_text) > 15:
                lines.append(f"      … ещё {len(missing_text) - 15}")
    lines.extend(
        [
            "",
            "ИТОГ: у всех файлов есть парный файл на диске (texts - audios)."
            if all_paired
            else "ИТОГ: есть файлы без пары на диске — исправьте каталог перед обучением/разметкой.",
        ]
    )

    print("\n".join(lines))
    return all_paired


def build_suffix_to_txt_basename(text_dir: Path) -> dict[str, str]:
    """Нижний регистр суффикса (после text) -> фактическое имя .txt на диске."""
    out: dict[str, str] = {}
    for p in sorted(text_dir.glob("*.txt")):
        suf = canonical_suffix(p.name)
        if suf is None:
            continue
        if suf not in out:
            out[suf] = p.name
    return out


def paired_text_parts_basename(
    audio_filename: str,
    suffix_to_txt: dict[str, str],
) -> object:
    """Имя .txt, парного к audio* (по суффиксу), или pd.NA если нет."""
    suf = canonical_suffix(audio_filename)
    if suf is None:
        return pd.NA
    name = suffix_to_txt.get(suf)
    return name if name is not None else pd.NA


def extract_excel_audio_id(filename: str) -> str | None:
    """Ключ строки метаданных в Excel (колонка audio file - audioNN)."""
    name = Path(filename).name
    m = re.search(r"(audio\d+)-\d+_urmi_labeled", name, re.I)
    if m:
        return m.group(1).lower()
    m = re.search(r"(audio\d+)_urmi_labeled", name, re.I)
    if m:
        return m.group(1).lower()
    return None


def extract_part_number(filename: str) -> int:
    m = re.search(r"audio\d+-(\d+)_urmi_labeled", Path(filename).name, re.I)
    if m:
        return int(m.group(1))
    if re.search(r"audio\d+_urmi_labeled\.[^.]+$", Path(filename).name, re.I):
        return 1
    return 0


def text_filename_to_excel_audio_id(filename: str) -> str | None:
    """Столбец audio_id в таблице: audioN из имени текстового файла/фрагмента."""
    stem = Path(filename).stem
    m = re.search(r"^text(\d+)-\d+_urmi_labeled$", stem, re.I)
    if m:
        return f"audio{m.group(1).lower()}"
    m = re.search(r"^text(\d+)_urmi_labeled$", stem, re.I)
    if m:
        return f"audio{m.group(1).lower()}"
    return None


def extract_text_basename_from_unlabeled_line(line: str) -> str | None:
    """Имя text…_urmi_labeled.txt из строки unlabeled_files.txt."""
    s = line.strip()
    if not s:
        return None
    m = re.search(r"(text\d+(?:-\d+)?_urmi_labeled)(?:\.txt)?", s, re.I)
    if not m:
        return None
    base = m.group(1).lower()
    return base if base.endswith(".txt") else f"{base}.txt"


def collect_audio_ids_forced_no_parts(
    unlabeled_path: Path, df: pd.DataFrame
) -> set[str]:

    out: set[str] = set()
    if not unlabeled_path.is_file():
        return out
    norm_sound = normalize_excel_text(SOUND_DICTIONARY_SOURCE_NENA)
    try:
        raw = unlabeled_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return out
    for line in raw.splitlines():
        bname = extract_text_basename_from_unlabeled_line(line)
        if bname is None:
            continue
        aid = text_filename_to_excel_audio_id(bname)
        if aid is None:
            continue
        mrows = df[df["audio_id"] == aid]
        if mrows.empty:
            continue
        for _, erow in mrows.iterrows():
            if normalize_excel_text(erow.get("source")) != norm_sound:
                out.add(aid)
                break
    return out


def format_duration_hms(seconds: float) -> str:
    total_seconds = int(round(max(0.0, seconds)))
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h}:{m:02d}:{s:02d}"


def audio_num_from_audio_id(audio_id: object) -> int:
    try:
        if pd.isna(audio_id):
            return 10**9
    except (ValueError, TypeError):
        pass
    if audio_id is None:
        return 10**9
    m = re.search(r"audio(\d+)", str(audio_id), re.I)
    return int(m.group(1)) if m else 10**9


def load_unlabeled_audio_ids(unlabeled_path: Path) -> set[str]:
    """Все audio_id, упомянутые в unlabeled_files.txt (по имени текстового файла в строке)."""
    if not unlabeled_path.is_file():
        return set()
    result: set[str] = set()
    try:
        raw = unlabeled_path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return set()
    for line in raw.splitlines():
        bname = extract_text_basename_from_unlabeled_line(line)
        if bname is None:
            continue
        aid = text_filename_to_excel_audio_id(bname)
        if aid:
            result.add(aid)
    return result


def row_from_excel(
    row: pd.Series,
    audio_id_val: str,
    part_num: int,
    parts_filename: str,
    text_parts_basename: object,
    parts_length: str,
) -> dict:
    return {
        "audio_id": audio_id_val,
        "part_num": part_num,
        "title of the text": row.get("title of the text"),
        "audio file": row.get("audio file"),
        "audio file length": row.get("audio file length"),
        "source": row.get("source"),
        "dialect": row.get("dialect"),
        "parts": parts_filename,
        "text parts": text_parts_basename,
        "parts' length": parts_length,
    }


def row_standalone_audio(
    audio_id_val: str | float,
    part_num: int,
    parts_filename: str,
    text_parts_basename: object,
    parts_length: str,
) -> dict:
    return {
        "audio_id": audio_id_val,
        "part_num": part_num,
        "title of the text": pd.NA,
        "audio file": pd.NA,
        "audio file length": pd.NA,
        "source": pd.NA,
        "dialect": pd.NA,
        "parts": parts_filename,
        "text parts": text_parts_basename,
        "parts' length": parts_length,
    }


def read_input_excel(excel_path: Path, sheet: str) -> pd.DataFrame:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    df.columns = [str(c).strip() for c in df.columns]
    if "audio file" not in df.columns:
        raise KeyError(f"Нет столбца 'audio file'. Есть: {list(df.columns)}")
    df = df[df["audio file"].notna()].copy()
    extracted = df["audio file"].astype(str).str.extract(r"(audio\d+)", expand=False)
    df["audio_id"] = extracted.str.lower()
    return df


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--excel", type=str, default="input.xlsx", help="Путь к input.xlsx")
    parser.add_argument(
        "--sheet",
        type=str,
        default="Urmi labeled",
        help="Имя листа в Excel",
    )
    parser.add_argument("--work-dir", type=str, default="data", help="Рабочая папка (Colab/data)")
    parser.add_argument(
        "--extract-subdir",
        type=str,
        default="unzipped",
        help="Подкаталог с audios/texts после распаковки (WORK_DIR/EXTRACT_SUBDIR)",
    )
    parser.add_argument(
        "--skip-download",
        action="store_true",
        help="Не скачивать archive.zip по ссылке (ожидайте уже загруженный архив или явные каталоги audios/texts)",
    )
    parser.add_argument(
        "--skip-unzip",
        action="store_true",
        help="Не распаковывать archive.zip даже если он есть на диске",
    )
    parser.add_argument(
        "--audio-dir",
        type=str,
        default=None,
        help="Явная папка audios (если уже есть на диске, без zip/unzipped)",
    )
    parser.add_argument(
        "--text-dir",
        type=str,
        default=None,
        help="Явная папка texts",
    )
    parser.add_argument(
        "--unlabeled-file",
        type=str,
        default=None,
        help="Явный путь к unlabeled_files.txt",
    )
    parser.add_argument(
        "--archive",
        type=str,
        default="archive.zip",
        help="ZIP после скачивания",
    )
    parser.add_argument(
        "--yandex-url",
        type=str,
        default="YANDEX LINK",
        help="Публичная ссылка Yandex Disk",
    )
    parser.add_argument(
        "--out",
        type=str,
        default="new_result.xlsx",
        help="Итоговый Excel",
    )
    args, leftover = parser.parse_known_args()
    leftover = strip_leading_jupyter_kernel_args(leftover)
    if leftover:
        print(
            "Предупреждение: неизвестные аргументы (игнорируются): "
            + " ".join(leftover),
            file=sys.stderr,
        )

    work_dir = Path(args.work_dir)
    extract_root = work_dir / args.extract_subdir
    excel_path = Path(args.excel)
    archive_path = Path(args.archive)

    explicit_audio = args.audio_dir
    explicit_text = args.text_dir
    if (explicit_audio is None) ^ (explicit_text is None):
        raise SystemExit(
            "Укажите оба аргумента --audio-dir и --text-dir, либо не указывайте ни одного."
        )

    print(f"In Colab: {in_colab()}")

    if explicit_audio is not None:
        audio_dir = Path(explicit_audio).expanduser().resolve()
        text_dir = Path(explicit_text).expanduser().resolve()
        unlabeled_path = (
            Path(args.unlabeled_file).expanduser().resolve()
            if args.unlabeled_file
            else (audio_dir.parent / "unlabeled_files.txt")
        )
    else:
        audio_dir = extract_root / "audios"
        text_dir = extract_root / "texts"
        unlabeled_path = (
            Path(args.unlabeled_file).expanduser().resolve()
            if args.unlabeled_file
            else (extract_root / "unlabeled_files.txt")
        )
        extract_root.mkdir(parents=True, exist_ok=True)

        if not args.skip_download:
            if not archive_path.is_file():
                print("Скачивание архива…")
                download_yandex_public(args.yandex_url, archive_path)

        if archive_path.is_file() and not args.skip_unzip:
            print("Распаковка zip…")
            with zipfile.ZipFile(archive_path, "r") as zip_ref:
                zip_ref.extractall(extract_root)

    if not audio_dir.is_dir() or not text_dir.is_dir():
        raise FileNotFoundError(
            f"Ожидались каталоги:\n  {audio_dir}\n  {text_dir}\n"
            "Проверьте структуру архива и --extract-subdir."
        )

    if not excel_path.is_file():
        raise FileNotFoundError(f"Нет Excel: {excel_path.resolve()}")

    df = read_input_excel(excel_path, args.sheet)
    suffix_to_txt = build_suffix_to_txt_basename(text_dir)

    unlabeled_audio_ids = load_unlabeled_audio_ids(unlabeled_path)
    forced_no_parts_audio_ids = collect_audio_ids_forced_no_parts(unlabeled_path, df)

    rows: list[dict] = []
    wav_files = sorted(p for p in audio_dir.iterdir() if p.suffix.lower() in AUDIO_EXTS)

    for filepath in tqdm(wav_files, desc="Аудио (librosa)"):
        filename = filepath.name
        audio_id_col = extract_excel_audio_id(filename)
        key = audio_id_col

        if key is not None and key in forced_no_parts_audio_ids:
            continue

        duration_str = NO_PARTS_LENGTH_ZERO
        try:
            y, sr = librosa.load(str(filepath), sr=None)
            duration_str = format_duration_hms(float(librosa.get_duration(y=y, sr=sr)))
        except Exception as e:
            print(f"Не удалось измерить длительность: {filename} | {e}")

        part_num = extract_part_number(filename)

        match_df = df[df["audio_id"] == key] if key is not None else pd.DataFrame()
        text_bn = paired_text_parts_basename(filename, suffix_to_txt)

        if not match_df.empty:
            for _, row in match_df.iterrows():
                rows.append(
                    row_from_excel(
                        row, key or "", part_num, filename, text_bn, duration_str
                    )
                )
        else:
            rows.append(
                row_standalone_audio(
                    key if key is not None else pd.NA,
                    part_num,
                    filename,
                    text_bn,
                    duration_str,
                )
            )

    existing_audio_ids_from_wav = {
        str(r["audio_id"]).strip().lower()
        for r in rows
        if pd.notna(r.get("audio_id"))
    }

    norm_sound = normalize_excel_text(SOUND_DICTIONARY_SOURCE_NENA)

    forced_no_parts_rows_added = 0
    for aid in sorted(forced_no_parts_audio_ids):
        qualifying = df[df["audio_id"] == aid]
        qualifying = qualifying[
            qualifying["source"].map(normalize_excel_text) != norm_sound
        ]
        for _, erow in qualifying.iterrows():
            rows.append(
                {
                    "audio_id": aid,
                    "part_num": 998,
                    "title of the text": erow.get("title of the text"),
                    "audio file": erow.get("audio file"),
                    "audio file length": erow.get("audio file length"),
                    "source": erow.get("source"),
                    "dialect": erow.get("dialect"),
                    "parts": NO_PARTS_PLACEHOLDER,
                    "text parts": NO_PARTS_PLACEHOLDER,
                    "parts' length": NO_PARTS_LENGTH_ZERO,
                }
            )
            forced_no_parts_rows_added += 1

    for _, row in df.iterrows():
        aid = row["audio_id"]
        aid_s = aid.strip().lower() if isinstance(aid, str) else ""
        if not aid_s or aid_s not in unlabeled_audio_ids:
            continue
        if aid_s in forced_no_parts_audio_ids:
            continue
        if aid_s not in existing_audio_ids_from_wav:
            if normalize_excel_text(row.get("source")) == norm_sound:
                continue
            rows.append(
                {
                    "audio_id": aid_s,
                    "part_num": 999,
                    "title of the text": row.get("title of the text"),
                    "audio file": row.get("audio file"),
                    "audio file length": row.get("audio file length"),
                    "source": row.get("source"),
                    "dialect": row.get("dialect"),
                    "parts": NO_PARTS_PLACEHOLDER,
                    "text parts": NO_PARTS_PLACEHOLDER,
                    "parts' length": NO_PARTS_LENGTH_ZERO,
                }
            )

    result_df = pd.DataFrame(rows)
    result_df["audio_num"] = result_df["audio_id"].apply(audio_num_from_audio_id)

    result_df = result_df.sort_values(
        by=["audio_num", "part_num", "parts"],
        kind="stable",
    )
    cols_out = [
        "audio_id",
        "title of the text",
        "audio file",
        "audio file length",
        "source",
        "dialect",
        "parts",
        "text parts",
        "parts' length",
    ]
    for c in cols_out:
        if c not in result_df.columns:
            result_df[c] = pd.NA
    export_df = result_df[cols_out].copy()

    out_path = Path(args.out)
    export_df.to_excel(out_path, index=False)

    n_aud = sum(1 for r in rows if r.get("parts") != NO_PARTS_PLACEHOLDER)
    n_syn = sum(1 for r in rows if r.get("parts") == NO_PARTS_PLACEHOLDER)
    txt_files_count = sum(1 for _ in text_dir.glob("*.txt"))
    _tp = export_df["text parts"]
    missing_txt_col = int(
        ((_tp.isna()) & (export_df["parts"] != NO_PARTS_PLACEHOLDER)).sum()
    )
    print(
        f"Сохранено: {out_path.resolve()}\n"
        f"   Строк по фрагментам аудио: {n_aud}; строк «no parts» (источники из unlabeled): "
        f"{n_syn}; из них после unlabeled_files+Excel: {forced_no_parts_rows_added}\n"
        f"   Строк всего: {len(rows)}; .txt на диске: {txt_files_count}\n"
        f"   Строк с фрагментами аудио, но без «text parts», кроме no parts: "
        f"{missing_txt_col}"
    )

    print("")
    check_pairs_report(
        text_dir,
        audio_dir,
        heading="Финальная проверка пар texts/ - audios/",
    )


if __name__ == "__main__":
    main()